# Tier 2 Assignment 1: Sequences and Alignment

**Covers**: Biological databases, BioPython, pairwise alignment, BLAST, MSA, phylogenetics

---

## Grading Rubric

| Problem | Points | Difficulty |
|---------|--------|------------|
| 1. Database Retrieval | 10 | (1 star) |
| 2. FASTA Statistics | 10 | (1 star) |
| 3. Dot Plot | 15 | (2 stars) |
| 4. Pairwise Alignment Scoring | 15 | (2 stars) |
| 5. BLAST Parser | 15 | (2 stars) |
| 6. MSA Conservation | 15 | (2 stars) |
| 7. Distance Matrix | 10 | (3 stars) |
| 8. Simple UPGMA | 10 | (3 stars) |
| **Total** | **100** | |


<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
The methods here are applied in sequence analysis, annotation, motif/protein interpretation, and evidence building from biological databases.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Similarity is not identity: alignments, scores, and biological function are related but not equivalent.
- Database provenance matters: always track which release/version generated your result.
- Threshold choices (e-value, identity, score) strongly change sensitivity vs specificity.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


In [ ]:
# Install dependencies if needed
# !pip install biopython

from Bio import Entrez, SeqIO
from Bio import pairwise2
from Bio.Blast import NCBIXML
import io

---

## Problem 1: Database Retrieval (1 star)

Use BioPython's `Entrez` module to fetch the GenBank record for human insulin
(accession **NM_000207**). Extract and print:
- Organism
- Description
- Sequence length
- First 100 nucleotides

**Example output**:
```
Organism: Homo sapiens
Description: Homo sapiens insulin (INS), mRNA
Length: 465 bp
First 100 nt: AGCCCTCCAGGACAGGCTGCATCAGAAGAGGCCATCAAGCAGATCACTGTCCTTCTGCCA...
```

**Grading**: 10 points — 3 pts correct fetch, 7 pts all four fields extracted and printed

In [ ]:
def fetch_genbank_record(accession: str, email: str) -> dict:
    """
    Fetch a GenBank record using Entrez and extract key fields.

    Args:
        accession: GenBank accession number (e.g. 'NM_000207')
        email: Your email address (required by NCBI)

    Returns:
        dict with keys: organism, description, length, first_100_nt
    """
    # YOUR CODE HERE
    pass


# Test
result = fetch_genbank_record("NM_000207", "your.email@example.com")
for key, value in result.items():
    print(f"{key}: {value}")

---

## Problem 2: FASTA Statistics (1 star)

Write a function that reads a FASTA file and returns a summary dictionary.

**Required fields**:
- `num_sequences`: total number of sequences
- `avg_length`: average sequence length (float, 2 decimal places)
- `min_length`: minimum sequence length
- `max_length`: maximum sequence length
- `avg_gc`: average GC content across all sequences (float, 2 decimal places)

**Example output** (for a 3-sequence FASTA):
```
num_sequences: 3
avg_length: 245.33
min_length: 180
max_length: 320
avg_gc: 52.41
```

**Grading**: 10 points — 2 pts per correct field

In [ ]:
def fasta_statistics(fasta_path: str) -> dict:
    """
    Compute summary statistics for sequences in a FASTA file.

    Args:
        fasta_path: Path to a FASTA file

    Returns:
        dict with keys: num_sequences, avg_length, min_length,
                        max_length, avg_gc
    """
    # YOUR CODE HERE
    pass


# Test with inline FASTA string written to a temp file
import tempfile, os

sample_fasta = """>seq1 Human insulin
AGCCCTCCAGGACAGGCTGCATCAGAAGAGGCCATCAAGCAGATCACTGTCCTTCTGCCATGGCCCTGTGG
>seq2 Mouse insulin
AGCCCTCCAGGACAGGCTGCATCAGAAGAGGCCATCAAGCAGATCACTGTCCTTCTGCCATGGCCCTGTGG
ATGAGCCTCCAGGACAGGCTGCATCAGAAGAGGCC
>seq3 Rat insulin
ATGAGCCTCCAGGACAGGCTGCATCAGAAGAGGCCATCAAGCAGATCACT
"""

with tempfile.NamedTemporaryFile(mode='w', suffix='.fasta', delete=False) as f:
    f.write(sample_fasta)
    tmp_path = f.name

stats = fasta_statistics(tmp_path)
for key, value in stats.items():
    print(f"{key}: {value}")

os.unlink(tmp_path)

---

## Problem 3: Dot Plot (2 stars)

Implement a text-based dot plot that compares two DNA sequences. A '.' is printed
where the sequences match within a sliding window.

**Parameters**:
- `seq1`, `seq2`: DNA strings to compare
- `window`: window size for noise reduction (default 1 = exact match)
  - A position (i, j) is marked if the fraction of matches in
    `seq1[i:i+window]` vs `seq2[j:j+window]` equals 1.0

**Example** (window=1, 10-char sequences):
```
    ATGCATGCAT
  A . . . . .
  T . . . . .
  ...
```

**Grading**: 15 points — 5 pts correct window logic, 5 pts matrix construction,
5 pts formatted output

In [ ]:
def dot_plot(seq1: str, seq2: str, window: int = 1) -> None:
    """
    Print a text-based dot plot comparing two DNA sequences.

    A position (i, j) is marked with '.' if seq1[i:i+window] == seq2[j:j+window].
    Positions outside the window boundary are left blank.

    Args:
        seq1: First DNA sequence (rows)
        seq2: Second DNA sequence (columns)
        window: Window size for comparison (default 1)
    """
    # YOUR CODE HERE
    pass


# Test
s1 = "ATGCATGC"
s2 = "ATGCTTGC"
print("Window=1:")
dot_plot(s1, s2, window=1)
print("\nWindow=3:")
dot_plot(s1, s2, window=3)

---

## Problem 4: Pairwise Alignment Scoring (2 stars)

Given two pre-aligned sequences (gaps represented as `'-'`), compute alignment
statistics. Assume a simple scoring scheme: a match is when both positions have
the same non-gap character; a gap is when either position has `'-'`; a mismatch
is everything else.

For **percent similarity**, count conserved substitutions using the BLOSUM-inspired
groups below (only for protein; for DNA, similarity == identity):
```
Groups: [ILVM], [FYW], [KRH], [DE], [STNQ], [AG], [CP]
```

**Example**:
```
seq1: ATCG-ATCG
seq2: ATCGATC-G
matches: 7, mismatches: 0, gaps: 2, identity: 87.5%, similarity: 87.5%
```

**Grading**: 15 points — 3 pts each for matches, mismatches, gaps, percent identity,
percent similarity

In [ ]:
def score_alignment(aligned_seq1: str, aligned_seq2: str) -> dict:
    """
    Score a pairwise alignment of two sequences.

    Args:
        aligned_seq1: First aligned sequence, gaps as '-'
        aligned_seq2: Second aligned sequence, gaps as '-'
            Both sequences must have equal length.

    Returns:
        dict with keys:
            matches (int), mismatches (int), gaps (int),
            percent_identity (float), percent_similarity (float)
    """
    # YOUR CODE HERE
    pass


# Test
a1 = "ATCG-ATCG"
a2 = "ATCGATC-G"
result = score_alignment(a1, a2)
print(f"Alignment:")
print(f"  {a1}")
print(f"  {a2}")
for key, value in result.items():
    print(f"{key}: {value}")

---

## Problem 5: BLAST Parser (2 stars)

Parse BLAST XML output using `Bio.Blast.NCBIXML` and return a filtered, sorted
hit list.

**Requirements**:
- Keep only hits with e-value < 0.001
- For each hit, extract: `accession`, `description`, `e_value`, `percent_identity`,
  `alignment_length`
- Return sorted by e-value (ascending)

**Note**: `percent_identity = hsp.identities / hsp.align_length * 100`
Use the first HSP of each alignment.

**Grading**: 15 points — 3 pts each field, 3 pts filtering, 3 pts sorting

*(A minimal BLAST XML string is provided below for testing.)*

In [ ]:
def parse_blast_xml(xml_source) -> list[dict]:
    """
    Parse BLAST XML output and return filtered hits.

    Args:
        xml_source: File path (str) or file-like object containing BLAST XML

    Returns:
        List of dicts sorted by e_value (ascending), each with keys:
            accession, description, e_value, percent_identity, alignment_length
        Only hits with e_value < 0.001 are included.
    """
    # YOUR CODE HERE
    pass


# Minimal BLAST XML for testing
BLAST_XML_SAMPLE = """<?xml version="1.0"?>
<!DOCTYPE BlastOutput PUBLIC "-//NCBI//NCBI BlastOutput/EN"
    "http://www.ncbi.nlm.nih.gov/dtd/NCBI_BlastOutput.dtd">
<BlastOutput>
  <BlastOutput_program>blastn</BlastOutput_program>
  <BlastOutput_iterations>
    <Iteration>
      <Iteration_hits>
        <Hit>
          <Hit_accession>NM_000207</Hit_accession>
          <Hit_def>Homo sapiens insulin (INS), mRNA</Hit_def>
          <Hit_hsps>
            <Hsp>
              <Hsp_evalue>1e-50</Hsp_evalue>
              <Hsp_identity>98</Hsp_identity>
              <Hsp_align-len>100</Hsp_align-len>
            </Hsp>
          </Hit_hsps>
        </Hit>
        <Hit>
          <Hit_accession>NM_001185097</Hit_accession>
          <Hit_def>Mus musculus insulin (Ins2), mRNA</Hit_def>
          <Hit_hsps>
            <Hsp>
              <Hsp_evalue>1e-30</Hsp_evalue>
              <Hsp_identity>85</Hsp_identity>
              <Hsp_align-len>100</Hsp_align-len>
            </Hsp>
          </Hit_hsps>
        </Hit>
        <Hit>
          <Hit_accession>XM_999999</Hit_accession>
          <Hit_def>Weak hit that should be filtered out</Hit_def>
          <Hit_hsps>
            <Hsp>
              <Hsp_evalue>0.5</Hsp_evalue>
              <Hsp_identity>40</Hsp_identity>
              <Hsp_align-len>80</Hsp_align-len>
            </Hsp>
          </Hit_hsps>
        </Hit>
      </Iteration_hits>
    </Iteration>
  </BlastOutput_iterations>
</BlastOutput>
"""

hits = parse_blast_xml(io.StringIO(BLAST_XML_SAMPLE))
print(f"Hits passing e-value filter: {len(hits)}")
for hit in hits:
    print(hit)

---

## Problem 6: Multiple Alignment Conservation (2 stars)

Given a multiple sequence alignment (list of aligned strings, all same length),
compute a conservation score at each position and find the most conserved positions.

**Conservation score** = fraction of the most common character at that column
(ignore gap `'-'` characters when computing the fraction).

**Return**: a list of `(position, score)` tuples for the top 10 most conserved
positions (1-based), sorted by score descending.

**Example**:
```
MSA:
  ACGT
  ACGT
  ACGG
Scores: [1.0, 1.0, 1.0, 0.67]
Top positions: [(1, 1.0), (2, 1.0), (3, 1.0), (4, 0.67)]
```

**Grading**: 15 points — 5 pts score calculation, 5 pts gap handling,
5 pts top-10 selection

In [ ]:
def conservation_scores(alignment: list[str]) -> list[float]:
    """
    Compute per-column conservation scores for a multiple sequence alignment.

    Conservation score at a position = count of most common non-gap character
    divided by total non-gap characters at that position.
    Returns 0.0 for all-gap columns.

    Args:
        alignment: List of aligned strings of equal length

    Returns:
        List of floats, one per alignment column
    """
    # YOUR CODE HERE
    pass


def top_conserved_positions(alignment: list[str], n: int = 10) -> list[tuple]:
    """
    Return the top n most conserved positions (1-based) with their scores.

    Args:
        alignment: List of aligned strings
        n: Number of top positions to return

    Returns:
        List of (position, score) tuples sorted by score descending
    """
    # YOUR CODE HERE
    pass


# Test
test_msa = [
    "ACGT-ACGT",
    "ACGT-ACGG",
    "ACGT-ACGA",
    "ACGG-ACGT",
    "ACGT-ACGT",
]
scores = conservation_scores(test_msa)
print("Conservation scores:", [round(s, 3) for s in scores])
print("Top conserved positions:", top_conserved_positions(test_msa, n=5))

---

## Problem 7: Distance Matrix (3 stars)

Given a set of DNA sequences (aligned to equal length), compute a pairwise
p-distance matrix.

**p-distance** between two sequences = proportion of positions that differ
(excluding positions where either sequence has a gap).

Print the result as a formatted table with sequence names as row/column headers.

**Example output**:
```
         seq1   seq2   seq3
seq1    0.000  0.125  0.250
seq2    0.125  0.000  0.200
seq3    0.250  0.200  0.000
```

**Grading**: 10 points — 4 pts correct p-distance formula, 3 pts symmetric matrix,
3 pts formatted output

In [ ]:
def p_distance(seq1: str, seq2: str) -> float:
    """
    Compute p-distance between two aligned sequences.

    Positions where either sequence has '-' are excluded.

    Args:
        seq1, seq2: Aligned sequences of equal length

    Returns:
        Proportion of differing non-gap sites (float in [0, 1])
    """
    # YOUR CODE HERE
    pass


def distance_matrix(sequences: dict[str, str]) -> None:
    """
    Compute and print a pairwise p-distance matrix.

    Args:
        sequences: Dict mapping sequence name to aligned sequence string
    """
    # YOUR CODE HERE
    pass


# Test sequences (pre-aligned, equal length)
test_seqs = {
    "Human":  "ATGCATGCATGCATGC",
    "Mouse":  "ATGCATGCATGCATGA",
    "Rat":    "ATGCATGCATGCATGG",
    "Zebrafish": "ATGCATGCTTGCATGC",
}
distance_matrix(test_seqs)

---

## Problem 8: Simple UPGMA (3 stars)

Implement the **UPGMA** (Unweighted Pair Group Method with Arithmetic Mean) algorithm
to build a phylogenetic tree from a distance matrix.

**Algorithm**:
1. Find the pair (i, j) with the minimum distance
2. Create a new node merging i and j; its branch length to each child = d(i,j)/2
3. Update distances: d(new, k) = (n_i * d(i,k) + n_j * d(j,k)) / (n_i + n_j)
4. Remove i and j; add the new node
5. Repeat until one node remains

Return the result as a **Newick format** string.

**Example** (for the distance matrix above):
```
((Human:0.03, Mouse:0.03):0.05, Rat:0.08):0.0;
```

**Grading**: 10 points — 4 pts correct merge logic, 3 pts distance update formula,
3 pts valid Newick output

In [ ]:
def upgma(names: list[str], dist: list[list[float]]) -> str:
    """
    Build a phylogenetic tree using the UPGMA algorithm.

    Args:
        names: List of taxon names
        dist:  Symmetric distance matrix as a list of lists
               (same ordering as names)

    Returns:
        Newick format string (with branch lengths, terminated by ';')
    """
    import copy

    # Work on mutable copies
    labels = list(names)
    matrix = [list(row) for row in dist]
    sizes = [1] * len(labels)   # number of original taxa in each cluster

    # YOUR CODE HERE
    pass


# Test — small 4-taxon example with known tree
taxa = ["A", "B", "C", "D"]
distances = [
    [0.0, 0.2, 0.4, 0.4],
    [0.2, 0.0, 0.4, 0.4],
    [0.4, 0.4, 0.0, 0.2],
    [0.4, 0.4, 0.2, 0.0],
]
tree = upgma(taxa, distances)
print("Newick tree:")
print(tree)

---

## Summary

In this assignment you practiced:

1. **Database Retrieval**: fetching records from NCBI via Entrez
2. **FASTA Statistics**: parsing and summarising sequence files
3. **Dot Plot**: a visual tool for sequence comparison
4. **Alignment Scoring**: quantifying alignment quality
5. **BLAST Parsing**: interpreting homology search results
6. **MSA Conservation**: identifying conserved positions
7. **Distance Matrix**: computing evolutionary distances
8. **UPGMA**: building a simple phylogenetic tree
